In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("diabetes_prediction_dataset.csv")
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["diabetes"].value_counts()

In [ ]:
df = df.drop_duplicates().dropna()
print("Dataset shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

In [ ]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X = pd.get_dummies(X, drop_first=True, dtype=int)
print(X.head())

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training:", X_train.shape, "Testing:", X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaling completed!")

In [ ]:
from sklearn.linear_model import Perceptron
model = Perceptron(max_iter=1000, tol=1e-3, random_state=42, class_weight="balanced")
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print("Perceptron training completed!")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("Accuracy:", round(accuracy_score(y_test, y_pred)*100, 2), "%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
weights = pd.Series(model.coef_[0], index=X.columns).sort_values().tail(12)
weights.plot(kind="barh", figsize=(9, 5))
plt.title("Perceptron Feature Weights")
plt.tight_layout()
plt.show()

In [ ]:
new_person = pd.DataFrame([{
    "gender": "Female", "age": 45, "hypertension": 0,
    "heart_disease": 0, "smoking_history": "never",
    "bmi": 27.5, "HbA1c_level": 5.8, "blood_glucose_level": 140
}])
new_person = pd.get_dummies(pd.concat([df.drop(columns=["diabetes"]), new_person], ignore_index=True), drop_first=True, dtype=int).tail(1)
new_person = new_person.reindex(columns=X.columns, fill_value=0)
print("Predicted diabetes class:", model.predict(scaler.transform(new_person))[0])
print("Note: Educational demonstration, not a medical diagnosis.")